Follow instructions here to clone and start Spark for exercise.
                                                  
**[Clone and start containers](https://github.com/josephmachado/data_storage_pattern/tree/main?tab=readme-ov-file#setup)**

## Setup

In [1]:
# Stop any existing session
try:
    spark.stop()
except:
    pass

26/03/26 15:51:27 ERROR TransportRequestHandler: Error sending result StreamResponse[streamId=/jars/iceberg-spark-runtime-4.0_2.13-1.10.0.jar,byteCount=47073384,body=FileSegmentManagedBuffer[file=/opt/spark/jars/iceberg-spark-runtime-4.0_2.13-1.10.0.jar,offset=0,length=47073384]] to /172.18.0.6:52076; closing connection
io.netty.channel.StacklessClosedChannelException
	at io.netty.channel.AbstractChannel.close(ChannelPromise)(Unknown Source)


In [2]:
chapter_name = "setup"

from pyspark.sql import SparkSession

# Default memory will not be sufficient for the examples below, so we up it to 4GB per 
# executor
executor_memory = "4g"
executor_cores = 4
num_executors = 2


spark = SparkSession.builder \
        .appName(chapter_name) \
        .master("spark://spark-master:7077") \
        .config("spark.executor.memory", executor_memory) \
        .config("spark.executor.cores", executor_cores) \
        .config("spark.executor.instances", num_executors) \
        .config("spark.cores.max", executor_cores * num_executors) \
        .getOrCreate()

Setting Spark log level to "WARN".


In [ ]:
# Create data and load into Iceberg tables for the code below
from pathlib import Path

from generate_data import run
from run_ddl import run_ddl 

#run(scaling_factor=1) # data will be 1GB on your disk
run_ddl(data_path=Path("./data"), spark=spark, recreate=True) # Load created data into Iceberg tables on Minio (S3 equivalent)

In [4]:
spark.sql("use prod.db") # this is the schema in which our data is

DataFrame[]

## Partitioning

In [ ]:
# FULL TABLE SCAN = All data is read into Spark Memory and then filtered
from pyspark.sql import functions as F

spark.read.table("lineitem").where(F.col("l_receiptdate") == "1992-01-05").explain()

In [5]:
# create a partitioned table
spark\
.read\
.table("lineitem")\
.writeTo("lineitem_rdp")\
.partitionedBy("l_receiptdate")\
.createOrReplace()

![partitioned lineitem](images/part_folder.png)

In [7]:
# PARTITION PRUNING = Only read the necessary partition
# No in memory filtering
from pyspark.sql import functions as F
spark\
.read\
.table("lineitem_rdp")\
.where(F.col("l_receiptdate") == "1992-01-05")\
.explain()

== Physical Plan ==
*(1) ColumnarToRow
+- BatchScan demo.prod.db.lineitem_rdp[l_orderkey#80, l_partkey#81, l_suppkey#82, l_linenumber#83, l_quantity#84, l_extendedprice#85, l_discount#86, l_tax#87, l_returnflag#88, l_linestatus#89, l_shipdate#90, l_commitdate#91, l_receiptdate#92, l_shipinstruct#93, l_shipmode#94, l_comment#95] demo.prod.db.lineitem_rdp (branch=null) [filters=l_receiptdate IS NOT NULL, l_receiptdate = 8039, groupedBy=] RuntimeFilters: []




In [ ]:
# Executing queries to see differences in the Spark UI at http://localhost:4040
spark\
.read\
.table("lineitem_rdp")\
.where(F.col("l_receiptdate") == "1992-01-05")\
.show(2)

spark\
.read\
.table("lineitem")\
.where(F.col("l_receiptdate") == "1992-01-05")\
.show(2)

![Non Partitioned v Partitioned](images/partition.png)

## Bucketing

In [ ]:
# NO BUCKETING GROUP BY = Exchange (movement of data between nodes in the cluster) 
# EXPENSIVE !!!
spark.sql("""
SELECT l_receiptdate
, sum(l_quantity) as total_quantity
FROM lineitem
GROUP BY l_receiptdate
""").explain()

In [9]:
# Create a bucketed table, with the bucketing based on the group by column
from pyspark.sql import functions as F

spark.read.table("lineitem")\
.writeTo("lineitem_bb_rd") \
.partitionedBy(  
    F.partitioning.bucket(4, "l_receiptdate")
).createOrReplace()

![Bucketed lineitem](images/bucket_folder.png)

In [10]:
# BUCKETING = NO Exchange 
# settings to ensure Spark + Iceberg can leverage the underlying data layout
spark.conf.set('spark.sql.sources.v2.bucketing.enabled','true')
spark.conf.set('spark.sql.iceberg.planning.preserve-data-grouping','true')

spark.sql("""
SELECT l_receiptdate
, sum(l_quantity) as total_quantity
FROM lineitem_bb_rd
GROUP BY l_receiptdate
""").explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[l_receiptdate#196], functions=[sum(l_quantity#188)])
   +- HashAggregate(keys=[l_receiptdate#196], functions=[partial_sum(l_quantity#188)])
      +- BatchScan demo.prod.db.lineitem_bb_rd[l_quantity#188, l_receiptdate#196] demo.prod.db.lineitem_bb_rd (branch=null) [filters=, groupedBy=l_receiptdate_bucket] RuntimeFilters: []




## Sorting

In [ ]:
# Can only be noticed if we run the query
# NO SORTING = Large data read into Spark Memory
spark.sql("""
select *
from lineitem
where l_extendedprice between 3000 and 10000
""")\
.write\
.csv(f"./{chapter_name}/lineitem", mode = 'overwrite')

In [ ]:
# Create Sorted Table
spark.read.table("lineitem")\
.sort("l_extendedprice")\
.coalesce(3)\
.writeTo("lineitem_sb_ep")\
.createOrReplace()

In [ ]:
# SORTING = Smaller data read into Spark Memory
spark.sql("""
select *
from lineitem_sb_ep
where l_extendedprice between 3000 and 10000
""")\
.write\
.csv(f"./{chapter_name}/lineitem_0rd", mode = 'overwrite')

![Sorted v UnSorted Reads](images/sorting.png)